# Running GOOD on the California example

This notebook loads the aggregated California graph (six balancing regions), adds flexible EV charging,
and optimizes dispatch and capacity expansion for three days starting at hour 4,608 (12 July) with state
renewable portfolio standards. It then repeats the run for a week with custom policies.

Units are MW, MWh, hours and $. See `docs/units.md`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import good
from good import visualization as vis

## Load the graph and policies

In [ ]:
graph = good.graph.graph_from_json('data/California.json.gz')
policies = good.utilities.read_json('data/policies.json')

print(f"{graph.number_of_nodes()} regions, {sum(len(d['assets']) for _, d in graph.nodes(data=True))} assets")

## Add EV charging

Each region gets an EV load of up to 2,000 MW that follows a random profile. Half of it can be shifted by up to
8 hours in either direction. Loads take positive MW of demand.

In [ ]:
rng = np.random.default_rng(0)

for region, node in graph.nodes(data=True):

    node['profiles'][f'{region}:ev'] = rng.random(8760).tolist()
    node['assets'][f'ev_load_{region}'] = {
        '_class': 'Load',
        'type': 'load',
        'installed_capacity': 2000,
        'profile': f'{region}:ev',
        'flex_capacity': 1000,
        'flex_max_delay': 8,
        'flex_max_advance': 8,
    }

## Build and solve a 3-day run

`steps=(4608, 4680)` selects hours 4,608 to 4,679 of the year: 12 to 14 July. HiGHS is the default solver.
Capacity costs are annualized and scaled by the modeled fraction of a year.

In [ ]:
start = 4608  # hour of the year: 12 July, 00:00

network = good.Network(steps=(start, start + 24 * 3), verbose=True).from_graph(graph, policies)
network.build()
network.size()

In [ ]:
network.solve()

solution = network.solution_graph()
dataframe = network.solution_dataframe(solution)

print(f'Objective: ${network.objective_value:,.0f}')

## Save results (the `outputs/` folder is not tracked by git)

In [ ]:
import os

os.makedirs('outputs', exist_ok=True)
good.graph.graph_to_json(solution, 'outputs/solution.json.gz')
dataframe.to_csv('outputs/solution.csv')

## Plot results

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
_ = vis.plot_lmps(ax, graph, solution)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(8, 10))
_ = vis.plot_demand(ax[0], graph, solution)
_ = vis.plot_total_generation(ax[1], graph, solution)
_ = vis.plot_net_generation(ax[2], graph, solution)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
_ = vis.plot_generation_by_fuel(ax, graph, solution)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
_ = vis.plot_new_capacity(ax, graph, solution)

In [ ]:
fig = vis.plot_dispatch_fuel_mix(graph, solution)

## Custom policies for a week

Policies select assets with attribute filters. Here: an 85% renewable share for California generation,
at least 70 GW of California wind and solar capacity, and a 10% reserve margin in which each asset counts
its `capacity_credit`.

In [ ]:
custom = {
    'california_ps': {
        '_class': 'Portfolio_Standard',
        'ratio': 0.85,
        'include': {'renewable': True, 'jurisdiction': 'CA', '_class': 'Producer'},
        'exclude': {'renewable': {'not': True}, 'jurisdiction': 'CA', '_class': 'Producer'},
    },
    'california_ct': {
        '_class': 'Capacity_Target',
        'target': 70_000,
        'include': {'type': ['solar', 'wind'], 'jurisdiction': 'CA'},
    },
    'california_rm': {
        '_class': 'Reserve_Margin',
        'margin': 0.10,
        'demand': {'_class': 'Load'},
        'supply': {'_class': {'not': 'Load'}},
    },
}

network = good.Network(steps=(start, start + 24 * 7), verbose=True).from_graph(graph, custom)
network.build()
network.solve()
solution = network.solution_graph()

print(f'Objective: ${network.objective_value:,.0f}')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
_ = vis.plot_generation_by_fuel(ax, graph, solution)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
_ = vis.plot_new_capacity(ax, graph, solution)